# Recomendar es factorizar

**Matemática 4 — Facultad de Informática, UNLP**

Este notebook implementa la factorización $R \approx U V^\top$ de la matriz usuario–película de
**MovieLens latest-small** con dos métodos:

- **ALS** (mínimos cuadrados alternados), que resuelve cada fila con las ecuaciones normales;
- **descenso de gradiente** completo sobre la suma de cuadrados del error.

Después de compararlos, el recomendador final usa el **modelo con sesgos**
$\hat r_{ij} = \mu + b_i + c_j + u_i \cdot v_j$, entrenado con ALS. Fue el que dio mejor resultado en los
experimentos: reparte las recomendaciones entre muchas más películas en vez de recomendarles
las mismas a todos.

Al final, la sección 15 evalúa ALS y descenso de gradiente separando las calificaciones en
entrenamiento y prueba, y dibuja la convergencia y las películas en el plano de $V$.

Todo el álgebra está escrita con numpy, sin librerías que factoricen o recomienden.

**Cómo correrlo:** *Entorno de ejecución → Ejecutar todo*. La primera vez descarga MovieLens (unos 1 MB)
y el notebook completo tarda menos de un minuto.

## 1. Instalación

Colab ya trae numpy y matplotlib; la celda solo se asegura de que estén. No hace falta nada más.

In [ ]:
!pip install -q numpy matplotlib

## 2. Importaciones y parámetros

Los hiperparámetros quedan todos juntos en esta celda. Se pueden cambiar y volver a ejecutar el
notebook desde acá.

| Parámetro | Valor | Por qué |
|---|---|---|
| `UMBRAL` | 40 | Mínimo de calificaciones por usuario y por película. Es el menor umbral con el que ALS y descenso de gradiente dan estimaciones razonables fuera de Ω (máximo $\lvert\hat r\rvert$ menor a 7). |
| `K` | 2 | Dimensión latente del modelo sin sesgos. Con k = 3 o 5, sin regularización, ALS da estimaciones absurdas fuera de Ω. |
| `ETA` | 5e-4 | Tasa de aprendizaje. Con 1e-3 el descenso diverge; con 2e-4 tarda el doble. |
| `EPSILON` | 1.0 | Corte: $\lvert f(t+1) - f(t)\rvert < \varepsilon$. La SCE está en el orden de las decenas de miles. |
| `MAX_ITER` | 3000 | Tope de iteraciones; el descenso de gradiente corta antes, en unas 460. |
| `SEMILLA` | 42 | Solo para que los resultados sean reproducibles. |
| `K_SESGOS` | 20 | Dimensión latente del modelo con sesgos. Junto con λ, es el par con menor error de prueba en el barrido de $k \in \{2, 5, 10, 20\}$ y $\lambda \in \{1, 5, 10, 20\}$. |
| `LAMBDA_SESGOS` | 10 | Regularización del modelo con sesgos: frena los factores exagerados de las películas con pocas calificaciones. |
| `PROPORCION_PRUEBA` | 0.2 | Fracción de las calificaciones de cada usuario que se reserva para prueba en la sección 15. |
| `KS_BARRIDO` | (2, 5, 10) | Valores de k del barrido de la sección 15. |
| `N_POPULARES` | 40 | Películas más calificadas que se dibujan en el plano de V (sección 15). |

In [ ]:
import csv
import logging
import time
import zipfile
from dataclasses import dataclass
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
LOGGER = logging.getLogger("recomendar")

# Preprocesamiento y modelo
UMBRAL = 40
K = 2
ETA = 5e-4
EPSILON = 1.0
MAX_ITER = 3000
SEMILLA = 42
ESCALA_INICIALIZACION = 1.0  # U0 y V0 uniformes en [0, 1)

# Modelo con sesgos
K_SESGOS = 20
LAMBDA_SESGOS = 10.0

# Evaluación con entrenamiento y prueba (sección 15)
PROPORCION_PRUEBA = 0.2
KS_BARRIDO = (2, 5, 10)
N_POPULARES = 40  # películas del plano de V

# Recomendaciones
USUARIO = 1     # id de usuario tal como aparece en MovieLens
TOP_N = 10

# Datos
URL_MOVIELENS = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
DIRECTORIO_DATOS = Path("data")

## 3. Errores propios

Excepciones con mensajes que dicen qué falló y dónde (por ejemplo, qué fila de ALS dio un sistema
singular), en lugar de un error genérico de numpy.

In [ ]:
class ErrorFactorizacion(Exception):
    "Excepción base del notebook."


class ErrorDescargaDataset(ErrorFactorizacion):
    def __init__(self, url: str) -> None:
        super().__init__(
            f"no se pudo descargar MovieLens desde {url}. Descargalo a mano desde esa URL, "
            "subí el .zip a Colab y descomprimilo en data/."
        )


class UmbralInsuficienteError(ErrorFactorizacion):
    def __init__(self, umbral: int, k: int) -> None:
        super().__init__(
            f"el umbral del filtro ({umbral}) es menor que k ({k}): ALS necesita al menos "
            f"k={k} observaciones por fila para que el sistema k×k tenga solución única."
        )


class ErrorDatosInsuficientes(ErrorFactorizacion):
    def __init__(self, umbral: int) -> None:
        super().__init__(
            f"el filtro por umbral={umbral} eliminó todos los usuarios o todas las películas."
        )


class SistemaSingularError(ErrorFactorizacion):
    def __init__(self, fila: int, n_observados: int) -> None:
        super().__init__(
            f"las ecuaciones normales de la fila {fila} son singulares: "
            f"solo hay {n_observados} observaciones para resolver el sistema."
        )


class DivergenciaError(ErrorFactorizacion):
    def __init__(self, iteracion: int, eta: float) -> None:
        super().__init__(
            f"el descenso de gradiente diverge en la iteración {iteracion}: f dejó de ser "
            f"finito con eta={eta:.4g}; probá con un eta más chico."
        )


class UsuarioNoEncontradoError(ErrorFactorizacion):
    def __init__(self, usuario_id: int) -> None:
        super().__init__(
            f"el usuario {usuario_id} no está en el conjunto filtrado (no existe en MovieLens "
            "o el filtro por UMBRAL lo eliminó): probá con otro USUARIO o con un UMBRAL más chico."
        )


class CalificacionesNuevasError(ErrorFactorizacion):
    def __init__(self, detalle: str) -> None:
        super().__init__(f"calificaciones del usuario nuevo: {detalle}")

## 4. Descarga de MovieLens

El dataset no se incluye en el repositorio porque la licencia de GroupLens no permite
redistribuirlo. Se descarga del sitio oficial y se descomprime en `data/ml-latest-small/`.
Si la carpeta ya existe, no se vuelve a descargar.

In [ ]:
def descargar_movielens(directorio_destino: Path, url: str = URL_MOVIELENS) -> Path:
    "Descarga MovieLens latest-small en `directorio_destino` y lo descomprime."
    directorio_dataset = directorio_destino / "ml-latest-small"
    if directorio_dataset.exists():
        LOGGER.info("MovieLens ya está en %s, no se descarga de nuevo", directorio_dataset)
        return directorio_dataset

    directorio_destino.mkdir(parents=True, exist_ok=True)
    ruta_zip = directorio_destino / "ml-latest-small.zip"
    LOGGER.info("descargando MovieLens desde %s", url)
    try:
        urlretrieve(url, ruta_zip)
    except (URLError, OSError) as error:
        raise ErrorDescargaDataset(url) from error

    with zipfile.ZipFile(ruta_zip) as archivo_zip:
        archivo_zip.extractall(directorio_destino)
    ruta_zip.unlink()
    return directorio_dataset


RUTA_MOVIELENS = descargar_movielens(DIRECTORIO_DATOS)
print(sorted(p.name for p in RUTA_MOVIELENS.iterdir()))

## 5. La matriz R y la máscara M

`ratings.csv` tiene una calificación por línea (de 0,5 a 5). Con eso se arman:

- $R$, de $m \times n$: la calificación del usuario $i$ a la película $j$, con `np.nan` en los huecos.
  Los huecos **nunca** se rellenan con ceros ni con promedios.
- $M$, la máscara de $\Omega$: `M[i, j] = True` si el usuario $i$ calificó la película $j$.

Los ids de MovieLens se reindexan desde 0 para usarlos como índices de fila y columna.

In [ ]:
@dataclass
class ConjuntoCalificaciones:
    "R (con NaN en los huecos), M y los mapeos de id de MovieLens a índice."
    R: np.ndarray
    M: np.ndarray
    id_usuario_a_indice: dict[int, int]
    id_pelicula_a_indice: dict[int, int]


def cargar_calificaciones(ruta_archivo: Path) -> ConjuntoCalificaciones:
    "Lee ratings.csv y arma R y M, reindexando usuarios y películas desde 0."
    filas: list[tuple[int, int, float]] = []
    with ruta_archivo.open(encoding="utf-8", newline="") as archivo:
        for registro in csv.DictReader(archivo):
            filas.append((int(registro["userId"]), int(registro["movieId"]), float(registro["rating"])))

    ids_usuario = sorted({u for u, _, _ in filas})
    ids_pelicula = sorted({p for _, p, _ in filas})
    id_usuario_a_indice = {id_crudo: i for i, id_crudo in enumerate(ids_usuario)}
    id_pelicula_a_indice = {id_crudo: j for j, id_crudo in enumerate(ids_pelicula)}

    R = np.full((len(ids_usuario), len(ids_pelicula)), np.nan)
    M = np.zeros(R.shape, dtype=bool)
    for id_usuario, id_pelicula, calificacion in filas:
        i = id_usuario_a_indice[id_usuario]
        j = id_pelicula_a_indice[id_pelicula]
        R[i, j] = calificacion
        M[i, j] = True

    LOGGER.info("cargadas %d calificaciones de %d usuarios y %d películas",
                len(filas), R.shape[0], R.shape[1])
    return ConjuntoCalificaciones(R, M, id_usuario_a_indice, id_pelicula_a_indice)

### Filtro por cantidad mínima de calificaciones

Se eliminan los usuarios y las películas con menos de `UMBRAL` calificaciones. Hay que repetirlo
hasta que no cambie nada: sacar una película puede dejar a un usuario por debajo del umbral, y al
revés.

El umbral tiene que ser al menos $k$: en ALS cada fila resuelve un sistema $k \times k$, que necesita
al menos $k$ observaciones para tener solución única. Con exactamente $k$ la solución existe pero no
es razonable, por eso el umbral que usamos es bastante mayor.

In [ ]:
def filtrar_por_minimo(datos: ConjuntoCalificaciones, umbral: int, k: int) -> ConjuntoCalificaciones:
    "Elimina filas y columnas con menos de `umbral` calificaciones, en cascada."
    if umbral < k:
        raise UmbralInsuficienteError(umbral, k)

    filas = np.arange(datos.M.shape[0])
    columnas = np.arange(datos.M.shape[1])
    while True:
        submascara = datos.M[np.ix_(filas, columnas)]
        filas_nuevas = filas[submascara.sum(axis=1) >= umbral]
        columnas_nuevas = columnas[submascara.sum(axis=0) >= umbral]
        if len(filas_nuevas) == len(filas) and len(columnas_nuevas) == len(columnas):
            break
        if len(filas_nuevas) == 0 or len(columnas_nuevas) == 0:
            raise ErrorDatosInsuficientes(umbral)
        filas, columnas = filas_nuevas, columnas_nuevas

    indice_a_id_usuario = {i: id_crudo for id_crudo, i in datos.id_usuario_a_indice.items()}
    indice_a_id_pelicula = {j: id_crudo for id_crudo, j in datos.id_pelicula_a_indice.items()}
    M_filtrada = datos.M[np.ix_(filas, columnas)]

    LOGGER.warning("filtro por umbral=%d: se eliminaron %d usuarios, %d películas y %d calificaciones",
                   umbral, datos.M.shape[0] - len(filas), datos.M.shape[1] - len(columnas),
                   int(datos.M.sum() - M_filtrada.sum()))

    return ConjuntoCalificaciones(
        R=datos.R[np.ix_(filas, columnas)],
        M=M_filtrada,
        id_usuario_a_indice={indice_a_id_usuario[viejo]: nuevo for nuevo, viejo in enumerate(filas)},
        id_pelicula_a_indice={indice_a_id_pelicula[viejo]: nuevo for nuevo, viejo in enumerate(columnas)},
    )


def cargar_titulos(ruta_movies: Path) -> dict[int, str]:
    "Lee movies.csv: id de película de MovieLens → título."
    with ruta_movies.open(encoding="utf-8", newline="") as archivo:
        return {int(registro["movieId"]): registro["title"] for registro in csv.DictReader(archivo)}

In [ ]:
# El umbral se valida contra el k más grande que se va a usar (el del modelo con sesgos)
datos = filtrar_por_minimo(cargar_calificaciones(RUTA_MOVIELENS / "ratings.csv"), UMBRAL, max(K, K_SESGOS))
titulos_por_id = cargar_titulos(RUTA_MOVIELENS / "movies.csv")
titulos_por_indice = {j: titulos_por_id[id_pelicula] for id_pelicula, j in datos.id_pelicula_a_indice.items()}

R, M = datos.R, datos.M
m, n = R.shape
print(f"MovieLens filtrado (umbral={UMBRAL}): {m} usuarios, {n} películas, {int(M.sum())} calificaciones.")
print(f"Densidad de R: {M.mean():.1%} de las posiciones están en Ω.")

## 6. Lo que comparten los dos métodos

ALS y descenso de gradiente comparten solamente la predicción, la función a minimizar y la
inicialización. La lógica de actualización de cada uno está en su propia sección.

**Predicción**: $\hat R = U V^\top$, con $U \in \mathbb{R}^{m \times k}$ y $V \in \mathbb{R}^{n \times k}$.

**Función a minimizar**: la suma de cuadrados del error, **solo sobre $\Omega$**,

$$f(U, V) = \sum_{(i,j) \in \Omega} \left(r_{ij} - u_i \cdot v_j\right)^2 .$$

**Inicialización**: $U_0$ y $V_0$ con entradas uniformes en $[0, 1)$, generadas una sola
vez con la semilla, para que los dos métodos arranquen del mismo punto.

In [ ]:
def predecir(U: np.ndarray, V: np.ndarray) -> np.ndarray:
    "R̂ = U·Vᵀ."
    return U @ V.T


def sce(R: np.ndarray, M: np.ndarray, U: np.ndarray, V: np.ndarray) -> float:
    "Suma de cuadrados del error sobre Ω. Fuera de Ω no cuenta."
    error = np.where(M, R - predecir(U, V), 0.0)
    return float(np.sum(error**2))


def inicializar_factores(m: int, n: int, k: int, escala: float,
                         generador: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    "U0 (m×k) y V0 (n×k) uniformes en [0, escala)."
    return generador.uniform(0.0, escala, size=(m, k)), generador.uniform(0.0, escala, size=(n, k))


@dataclass
class ResultadoEntrenamiento:
    "Factores finales, valores de f por iteración (incluido f0), iteraciones, tiempo y motivo de corte."
    U: np.ndarray
    V: np.ndarray
    historial_f: list[float]
    n_iteraciones: int
    tiempo_segundos: float
    motivo_corte: str

## 7. ALS: mínimos cuadrados alternados

Con $V$ fija, $f$ se separa en un problema de mínimos cuadrados independiente para cada usuario.
Si $V_{\Omega_i}$ son las filas de $V$ de las películas que calificó el usuario $i$ y $r_{\Omega_i}$ sus
calificaciones, la fila $u_i$ sale de las **ecuaciones normales**:

$$\left(V_{\Omega_i}^\top V_{\Omega_i}\right) u_i = V_{\Omega_i}^\top r_{\Omega_i} .$$

El paso de $V$ es el mismo problema mirado desde las películas: es la **misma función**, llamada con
$R^\top$, $M^\top$ y $U$. Cada sistema se resuelve con `np.linalg.solve`.

ALS alterna los dos pasos hasta que $\lvert f(t+1) - f(t)\rvert < \varepsilon$.

In [ ]:
def resolver_factor(R: np.ndarray, M: np.ndarray, F: np.ndarray, lambda_: float = 0.0) -> np.ndarray:
    """Resuelve por fila las ecuaciones normales. Con (R, M, V) da U; con (R.T, M.T, U) da V.

    Con lambda_ > 0 resuelve las regularizadas (FᵀF + λI) x = Fᵀr, que usa el modelo con
    sesgos (sección 13). Con el default 0 son las ecuaciones normales sin regularizar.
    """
    filas, k = R.shape[0], F.shape[1]
    factor_nuevo = np.empty((filas, k))
    for i in range(filas):
        columnas_obs = np.where(M[i, :])[0]
        F_obs = F[columnas_obs, :]
        r_obs = R[i, columnas_obs]

        # Ecuaciones normales de mínimos cuadrados de la fila i (+ λI si hay regularización)
        A = F_obs.T @ F_obs + lambda_ * np.eye(k)
        b = F_obs.T @ r_obs
        try:
            factor_nuevo[i, :] = np.linalg.solve(A, b)
        except np.linalg.LinAlgError as error:
            raise SistemaSingularError(fila=i, n_observados=len(columnas_obs)) from error
    return factor_nuevo


def entrenar_als(R: np.ndarray, M: np.ndarray, U0: np.ndarray, V0: np.ndarray,
                 epsilon: float, max_iter: int) -> ResultadoEntrenamiento:
    "Alterna el paso de U y el de V hasta |f(t+1) − f(t)| < epsilon o max_iter."
    inicio = time.perf_counter()
    U, V = U0, V0
    historial_f = [sce(R, M, U, V)]
    motivo_corte = "max_iter"

    for _ in range(max_iter):
        # Paso de U y paso de V, con la misma función (relación inversa)
        U = resolver_factor(R, M, V)
        V = resolver_factor(R.T, M.T, U)
        historial_f.append(sce(R, M, U, V))
        if abs(historial_f[-1] - historial_f[-2]) < epsilon:
            motivo_corte = "tolerancia"
            break

    if motivo_corte == "max_iter":
        LOGGER.warning("ALS: se alcanzó max_iter=%d sin cortar por tolerancia", max_iter)
    n_iteraciones = len(historial_f) - 1
    LOGGER.info("ALS terminó: %d iteraciones (%s), f final=%.6g", n_iteraciones, motivo_corte, historial_f[-1])
    return ResultadoEntrenamiento(U, V, historial_f, n_iteraciones, time.perf_counter() - inicio, motivo_corte)

## 8. Descenso de gradiente

Con $E = M \odot (R - U V^\top)$, el error con ceros fuera de $\Omega$, los gradientes de $f$ son:

$$\nabla_U f = -2\, E\, V, \qquad \nabla_V f = -2\, E^\top U .$$

Es la versión **completa** (no estocástica): en cada iteración se usan todas las calificaciones.
Los dos gradientes se calculan con $U$ y $V$ de la iteración $t$ y se actualizan **juntos**:

$$U^{(t+1)} = U^{(t)} - \eta\, \nabla_U f\big(U^{(t)}, V^{(t)}\big), \qquad
V^{(t+1)} = V^{(t)} - \eta\, \nabla_V f\big(U^{(t)}, V^{(t)}\big).$$

Si se actualizara $U$ y después se calculara el gradiente de $V$ con la $U$ nueva, el método pasaría
a ser alternado, como ALS.

In [ ]:
def gradiente_sce(R: np.ndarray, M: np.ndarray, U: np.ndarray,
                  V: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    "∇_U f y ∇_V f de la SCE, los dos evaluados en el mismo (U, V)."
    E = np.where(M, R - predecir(U, V), 0.0)
    return -2 * E @ V, -2 * E.T @ U


def entrenar_gd(R: np.ndarray, M: np.ndarray, U0: np.ndarray, V0: np.ndarray,
                eta: float, epsilon: float, max_iter: int) -> ResultadoEntrenamiento:
    "Descenso de gradiente completo con actualización simultánea de U y V."
    inicio = time.perf_counter()
    U, V = U0, V0
    historial_f = [sce(R, M, U, V)]
    motivo_corte = "max_iter"

    for iteracion in range(1, max_iter + 1):
        grad_U, grad_V = gradiente_sce(R, M, U, V)
        # x(t+1) = x(t) − eta·∇f(x(t)), U y V con el gradiente de la iteración t
        U = U - eta * grad_U
        V = V - eta * grad_V

        f_siguiente = sce(R, M, U, V)
        if not np.isfinite(f_siguiente):
            raise DivergenciaError(iteracion, eta)
        if f_siguiente > historial_f[-1]:
            LOGGER.warning("GD: f aumentó en la iteración %d; eta=%.4g podría ser grande", iteracion, eta)
        historial_f.append(f_siguiente)
        if iteracion % 100 == 0:
            LOGGER.info("GD iteración %d: f=%.6g", iteracion, f_siguiente)
        if abs(historial_f[-1] - historial_f[-2]) < epsilon:
            motivo_corte = "tolerancia"
            break

    if motivo_corte == "max_iter":
        LOGGER.warning("GD: se alcanzó max_iter=%d sin cortar por tolerancia", max_iter)
    n_iteraciones = len(historial_f) - 1
    LOGGER.info("GD terminó: %d iteraciones (%s), f final=%.6g", n_iteraciones, motivo_corte, historial_f[-1])
    return ResultadoEntrenamiento(U, V, historial_f, n_iteraciones, time.perf_counter() - inicio, motivo_corte)

## 9. Entrenamiento

Se genera **un solo** par $U_0, V_0$ y cada método recibe su propia copia, así la comparación arranca
del mismo punto.

In [ ]:
generador = np.random.default_rng(SEMILLA)
U0, V0 = inicializar_factores(m, n, K, ESCALA_INICIALIZACION, generador)

resultado_als = entrenar_als(R, M, U0.copy(), V0.copy(), epsilon=EPSILON, max_iter=MAX_ITER)
resultado_gd = entrenar_gd(R, M, U0.copy(), V0.copy(), eta=ETA, epsilon=EPSILON, max_iter=MAX_ITER)

## 10. Comparación

Para cada método: cuántas iteraciones hizo, por qué cortó, cuánto tardó y el valor final de $f$.

In [ ]:
print(f"{'método':<10}{'iteraciones':>12}{'motivo de corte':>18}{'tiempo (s)':>12}{'SCE final':>14}")
for nombre, resultado in [("ALS", resultado_als), ("GD", resultado_gd)]:
    print(f"{nombre:<10}{resultado.n_iteraciones:>12}{resultado.motivo_corte:>18}"
          f"{resultado.tiempo_segundos:>12.4f}{resultado.historial_f[-1]:>14.4f}")

### Convergencia

$f$ en función de la iteración, en escala logarítmica. ALS baja casi todo en las primeras iteraciones,
porque cada paso resuelve un problema de mínimos cuadrados de forma exacta. El descenso de gradiente
avanza de a pasos chicos de tamaño $\eta$ y necesita muchas más iteraciones, aunque cada una es
más barata.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(resultado_als.historial_f, label="ALS")
ax.plot(resultado_gd.historial_f, label="Descenso de gradiente")
ax.set_yscale("log")
ax.set_xlabel("iteración")
ax.set_ylabel("f (SCE)")
ax.set_title("Convergencia sobre MovieLens filtrado")
ax.legend()
plt.show()

## 11. Recomendaciones del modelo sin sesgos

Para un usuario $i$, la estimación de cada película es $\hat r_{ij} = u_i \cdot v_j$. Se recomiendan
las `TOP_N` películas que **no** calificó, ordenadas por $\hat r_{ij}$. `USUARIO` es el id tal como
aparece en MovieLens.

In [ ]:
def recomendar_top_n(U: np.ndarray, V: np.ndarray, M: np.ndarray, indice_usuario: int,
                     cantidad: int) -> list[tuple[str, float]]:
    "Las `cantidad` películas no calificadas por el usuario con mayor r̂ᵢⱼ."
    r_hat = predecir(U, V)[indice_usuario, :]
    candidatos = [j for j in range(V.shape[0]) if not M[indice_usuario, j]]
    candidatos.sort(key=lambda j: r_hat[j], reverse=True)
    return [(titulos_por_indice[j], float(r_hat[j])) for j in candidatos[:cantidad]]


if USUARIO not in datos.id_usuario_a_indice:
    raise UsuarioNoEncontradoError(USUARIO)
indice_usuario = datos.id_usuario_a_indice[USUARIO]

recomendaciones_als = recomendar_top_n(resultado_als.U, resultado_als.V, M, indice_usuario, TOP_N)
recomendaciones_gd = recomendar_top_n(resultado_gd.U, resultado_gd.V, M, indice_usuario, TOP_N)

print(f"Top-{TOP_N} para el usuario {USUARIO} (calificó {int(M[indice_usuario].sum())} películas)\n")
print(f"{'#':>3}  {'ALS':<58}{'Descenso de gradiente'}")
for posicion, ((titulo_als, r_als), (titulo_gd, r_gd)) in enumerate(zip(recomendaciones_als, recomendaciones_gd), 1):
    print(f"{posicion:>3}  {titulo_als[:50]:<50} {r_als:5.2f}  {titulo_gd[:50]:<50} {r_gd:5.2f}")

## 12. Qué aprendió cada factor latente

Las dimensiones latentes no se etiquetan ni se inicializan con ningún significado: emergen del
entrenamiento. Para mirarlas, se listan las películas con los valores más altos y más bajos en cada
columna de $V$ (la de ALS). La interpretación queda para el análisis.

In [ ]:
V_als = resultado_als.V
for factor in range(V_als.shape[1]):
    orden = np.argsort(V_als[:, factor])
    print(f"Factor {factor}")
    print("  valores más altos:")
    for j in orden[::-1][:5]:
        print(f"    {V_als[j, factor]:7.3f}  {titulos_por_indice[j]}")
    print("  valores más bajos:")
    for j in orden[:5]:
        print(f"    {V_als[j, factor]:7.3f}  {titulos_por_indice[j]}")
    print()

## 13. El modelo con sesgos

### El problema: todos reciben las mismas recomendaciones

Con $\hat r_{ij} = u_i \cdot v_j$ y $k = 2$, ese único producto tiene que explicar todo a la vez: que el
promedio de las notas ronda 3,7, que hay usuarios generosos y exigentes, que hay películas que casi
todos califican alto y, recién al final, los gustos personales. Lo común es lo que más pesa en el
error, así que los factores se gastan en eso. Geométricamente, cada película es un punto del plano y
cada usuario una dirección; como todas las notas son positivas, casi todos los usuarios miran hacia el
mismo borde de la nube, y ganan siempre las mismas películas "buenas para todos".

### La solución: darle a cada efecto su propio término

$$\hat r_{ij} = \mu + b_i + c_j + u_i \cdot v_j$$

| Término | Qué captura |
|---|---|
| $\mu$ | El promedio de todas las calificaciones (fijo, no se entrena) |
| $b_i$ | Si el usuario $i$ califica más alto o más bajo que el promedio |
| $c_j$ | Qué tan bien califica todo el mundo a la película $j$ |
| $u_i \cdot v_j$ | Cuánto le gusta la película $j$ al usuario $i$ más allá de lo esperable |

Como la calidad general ahora va en $c_j$, $U$ y $V$ quedan libres para las diferencias de gusto.
La función a minimizar agrega regularización, que frena los factores exagerados de las películas con
pocas calificaciones:

$$f = \sum_{(i,j) \in \Omega} \left(r_{ij} - \hat r_{ij}\right)^2
+ \lambda \Big(\sum_i \lVert u_i \rVert^2 + \sum_j \lVert v_j \rVert^2 + \sum_i b_i^2 + \sum_j c_j^2\Big).$$

### ALS con sesgos

Con $V$ y $c$ fijos, las incógnitas del usuario $i$ son $(u_i, b_i) \in \mathbb{R}^{k+1}$. Si a $V$ se le
agrega una columna de unos, $[V \mid 1] \cdot (u_i, b_i) = u_i \cdot v_j + b_i$, y el paso es otra vez un
problema de mínimos cuadrados, con objetivo $r_{ij} - \mu - c_j$. Se resuelve con la **misma**
`resolver_factor` de la sección 7, con las ecuaciones normales regularizadas:

$$\left([V \mid 1]_{\Omega_i}^\top [V \mid 1]_{\Omega_i} + \lambda I\right) (u_i, b_i) = [V \mid 1]_{\Omega_i}^\top \left(r_{\Omega_i} - \mu - c_{\Omega_i}\right).$$

El paso de películas es el mismo con $[U \mid 1]$, objetivo $r_{ij} - \mu - b_i$ y la matriz transpuesta.
Con $\lambda > 0$ la matriz es definida positiva, así que el sistema siempre tiene solución única.

### Los dos ordenamientos

- **A, por $\hat r_{ij}$ completo:** incluye $c_j$, así que sigue favoreciendo a las películas que todos valoran.
- **B, solo por $u_i \cdot v_j$:** ordena por afinidad personal. $\mu$ y $b_i$ no cambian el orden de las
  películas de un usuario; lo que se saca es $c_j$.

B fue el ordenamiento que cumplió todos los criterios fijados de antemano en el experimento, y es el
que usa el recomendador de la sección 14. Tiene una contraprestación: en una partición
entrenamiento/prueba, la precisión@10 de B (0,1122) quedó apenas por encima de la del modelo sin
sesgos (0,1116), mientras que A acierta más (0,1530) porque recomendar lo popular es una apuesta
segura, aunque repetitiva.

In [ ]:
def predecir_con_sesgos(U: np.ndarray, V: np.ndarray, b: np.ndarray, c: np.ndarray,
                        mu: float) -> np.ndarray:
    "r̂ᵢⱼ = μ + bᵢ + cⱼ + Uᵢ·Vⱼ."
    return mu + b[:, None] + c[None, :] + predecir(U, V)


def f_sesgos(R: np.ndarray, M: np.ndarray, U: np.ndarray, V: np.ndarray, b: np.ndarray,
             c: np.ndarray, mu: float, lambda_: float) -> float:
    "SCE sobre Ω más λ por la suma de los cuadrados de U, V, b y c."
    error = np.where(M, R - predecir_con_sesgos(U, V, b, c, mu), 0.0)
    penalizacion = float(np.sum(U**2) + np.sum(V**2) + np.sum(b**2) + np.sum(c**2))
    return float(np.sum(error**2)) + lambda_ * penalizacion


def paso_aumentado(objetivo: np.ndarray, M: np.ndarray, F: np.ndarray,
                   lambda_: float) -> tuple[np.ndarray, np.ndarray]:
    "Resuelve (factor, sesgo) de cada fila con [F | 1] y `resolver_factor`. Devuelve los dos por separado."
    F_aumentada = np.hstack([F, np.ones((F.shape[0], 1))])
    solucion = resolver_factor(objetivo, M, F_aumentada, lambda_)
    return solucion[:, :-1], solucion[:, -1]


def paso_usuarios(R: np.ndarray, M: np.ndarray, V: np.ndarray, c: np.ndarray, mu: float,
                  lambda_: float) -> tuple[np.ndarray, np.ndarray]:
    "(U, b) con [V | 1] y objetivo rᵢⱼ − μ − cⱼ."
    return paso_aumentado(R - mu - c[None, :], M, V, lambda_)


def paso_peliculas(R: np.ndarray, M: np.ndarray, U: np.ndarray, b: np.ndarray, mu: float,
                   lambda_: float) -> tuple[np.ndarray, np.ndarray]:
    "(V, c) con [U | 1] y objetivo rᵢⱼ − μ − bᵢ: el mismo paso, con la matriz transpuesta."
    return paso_aumentado((R - mu - b[:, None]).T, M.T, U, lambda_)


@dataclass
class ResultadoSesgos:
    "Factores, sesgos y μ finales, valores de f por iteración (incluido f0), iteraciones, tiempo y motivo de corte."
    U: np.ndarray
    V: np.ndarray
    b: np.ndarray
    c: np.ndarray
    mu: float
    historial_f: list[float]
    n_iteraciones: int
    tiempo_segundos: float
    motivo_corte: str


def entrenar_als_sesgos(R: np.ndarray, M: np.ndarray, U0: np.ndarray, V0: np.ndarray,
                        epsilon: float, max_iter: int, lambda_: float) -> ResultadoSesgos:
    "ALS con sesgos: μ fijo, b = c = 0 al inicio, alterna el paso de usuarios y el de películas."
    inicio = time.perf_counter()
    mu = float(np.mean(R[M]))
    U, V = U0, V0
    b, c = np.zeros(R.shape[0]), np.zeros(R.shape[1])
    historial_f = [f_sesgos(R, M, U, V, b, c, mu, lambda_)]
    motivo_corte = "max_iter"

    for _ in range(max_iter):
        U, b = paso_usuarios(R, M, V, c, mu, lambda_)
        V, c = paso_peliculas(R, M, U, b, mu, lambda_)
        historial_f.append(f_sesgos(R, M, U, V, b, c, mu, lambda_))
        if abs(historial_f[-1] - historial_f[-2]) < epsilon:
            motivo_corte = "tolerancia"
            break

    if motivo_corte == "max_iter":
        LOGGER.warning("ALS con sesgos: se alcanzó max_iter=%d sin cortar por tolerancia", max_iter)
    n_iteraciones = len(historial_f) - 1
    LOGGER.info("ALS con sesgos terminó: %d iteraciones (%s), f final=%.6g",
                n_iteraciones, motivo_corte, historial_f[-1])
    return ResultadoSesgos(U, V, b, c, mu, historial_f, n_iteraciones, time.perf_counter() - inicio, motivo_corte)

### Entrenamiento con sesgos

$U_0$ y $V_0$ se inicializan igual que antes, ahora con `K_SESGOS` columnas; $b$ y $c$ arrancan en 0.
Se entrena sobre todas las calificaciones de $\Omega$.

In [ ]:
U0_sesgos, V0_sesgos = inicializar_factores(m, n, K_SESGOS, ESCALA_INICIALIZACION,
                                            np.random.default_rng(SEMILLA))
resultado_sesgos = entrenar_als_sesgos(R, M, U0_sesgos, V0_sesgos, epsilon=EPSILON,
                                       max_iter=MAX_ITER, lambda_=LAMBDA_SESGOS)

r_hat_sesgos = predecir_con_sesgos(resultado_sesgos.U, resultado_sesgos.V,
                                   resultado_sesgos.b, resultado_sesgos.c, resultado_sesgos.mu)
print(f"μ = {resultado_sesgos.mu:.4f}")
print(f"{resultado_sesgos.n_iteraciones} iteraciones ({resultado_sesgos.motivo_corte}), "
      f"{resultado_sesgos.tiempo_segundos:.2f} s, f final = {resultado_sesgos.historial_f[-1]:.4f}")
print(f"máximo |r̂| fuera de Ω: {np.abs(r_hat_sesgos[~M]).max():.2f}")

### Concentración: ¿a cuántas películas distintas llegan las recomendaciones?

Se arma el top-`TOP_N` de cada uno de los usuarios reales y se cuenta cuántas películas distintas
aparecen en algún top y en qué porcentaje de los tops está la más frecuente. Cuanto menos se repitan,
más personales son las recomendaciones.

In [ ]:
def tops_de_todos(puntajes: np.ndarray, M: np.ndarray, cantidad: int) -> np.ndarray:
    "Índices del top-`cantidad` de cada usuario, entre las películas que no calificó."
    puntajes_sin_calificadas = np.where(M, -np.inf, puntajes)
    return np.argsort(-puntajes_sin_calificadas, axis=1, kind="stable")[:, :cantidad]


def medir_concentracion(tops: np.ndarray) -> tuple[int, str, float]:
    "Películas distintas en algún top, la más frecuente y en qué fracción de los tops aparece."
    apariciones = np.bincount(tops.ravel(), minlength=n)
    mas_frecuente = int(np.argmax(apariciones))
    return len(np.unique(tops)), titulos_por_indice[mas_frecuente], apariciones[mas_frecuente] / tops.shape[0]


s = resultado_sesgos
puntajes_por_modelo = {
    f"sin sesgos (ALS, k={K})": predecir(resultado_als.U, resultado_als.V),
    "con sesgos, A (r̂ completo)": predecir_con_sesgos(s.U, s.V, s.b, s.c, s.mu),
    "con sesgos, B (u·v)": predecir(s.U, s.V),
}

print(f"Concentración de los top-{TOP_N} de los {m} usuarios\n")
print(f"{'modelo':<30}{'películas distintas':>21}   {'la más frecuente'}")
for nombre, puntajes in puntajes_por_modelo.items():
    distintas, titulo, fraccion = medir_concentracion(tops_de_todos(puntajes, M, TOP_N))
    print(f"{nombre:<30}{distintas:>12} de {n:<6}   {titulo[:40]}, {fraccion:.1%}")

### Recomendaciones con sesgos para `USUARIO`

El mismo usuario de la sección 11, con los dos ordenamientos. En B el número es $u_i \cdot v_j$: cuánto
más (o menos) que lo esperable le gustaría la película.

In [ ]:
puntajes_a = predecir_con_sesgos(s.U, s.V, s.b, s.c, s.mu)[indice_usuario]
puntajes_b = predecir(s.U, s.V)[indice_usuario]
no_calificadas = [j for j in range(n) if not M[indice_usuario, j]]
top_a = sorted(no_calificadas, key=lambda j: puntajes_a[j], reverse=True)[:TOP_N]
top_b = sorted(no_calificadas, key=lambda j: puntajes_b[j], reverse=True)[:TOP_N]

print(f"Top-{TOP_N} con sesgos para el usuario {USUARIO} (b = {s.b[indice_usuario]:+.3f})\n")
print(f"{'#':>3}  {'A: r̂ completo':<58}{'B: u·v (personal)'}")
for posicion, (j_a, j_b) in enumerate(zip(top_a, top_b), 1):
    print(f"{posicion:>3}  {titulos_por_indice[j_a][:50]:<50} {puntajes_a[j_a]:5.2f}  "
          f"{titulos_por_indice[j_b][:50]:<50} {puntajes_b[j_b]:+5.2f}")

## 14. Probalo con tus propias calificaciones

Un usuario nuevo es una fila más de $R$. Con el modelo con sesgos ya entrenado ($\mu$, $V$ y $c$ fijos),
su par $(u, b)$ sale de **un paso de usuarios de ALS con sesgos**: el mismo `paso_usuarios` de la
sección 13, ahora con una sola fila. Las recomendaciones se ordenan por la parte personal $u \cdot v_j$
(ordenamiento B); entre paréntesis está la estimación completa $\hat r_j$.

Abajo están las 30 películas más calificadas del conjunto filtrado. Editá el diccionario
`mis_calificaciones` con al menos 5 de ellas (notas de 1 a 5) y ejecutá la celda.

In [ ]:
MIN_CALIFICACIONES_NUEVO = 5
mas_calificadas = np.argsort(-M.sum(axis=0), kind="stable")[:30]
for j in mas_calificadas:
    print(titulos_por_indice[j])

In [ ]:
mis_calificaciones = {
    "Matrix, The (1999)": 5,
    "Toy Story (1995)": 4,
    "Independence Day (a.k.a. ID4) (1996)": 2,
    "Pulp Fiction (1994)": 5,
    "Forrest Gump (1994)": 3,
}

indice_por_titulo = {titulo: j for j, titulo in titulos_por_indice.items()}
if len(mis_calificaciones) < MIN_CALIFICACIONES_NUEVO:
    raise CalificacionesNuevasError(f"hacen falta al menos {MIN_CALIFICACIONES_NUEVO}, hay {len(mis_calificaciones)}")

r_nuevo = np.full((1, n), np.nan)
m_nuevo = np.zeros((1, n), dtype=bool)
for titulo, nota in mis_calificaciones.items():
    if titulo not in indice_por_titulo:
        raise CalificacionesNuevasError(f"'{titulo}' no está en el conjunto filtrado; copiá el título de la lista")
    if not 1 <= nota <= 5:
        raise CalificacionesNuevasError(f"la nota de '{titulo}' tiene que estar entre 1 y 5")
    r_nuevo[0, indice_por_titulo[titulo]] = nota
    m_nuevo[0, indice_por_titulo[titulo]] = True

# Paso de usuarios con sesgos para una sola fila, con μ, V y c del modelo fijos
s = resultado_sesgos
u_nuevo, b_nuevo = paso_usuarios(r_nuevo, m_nuevo, s.V, s.c, s.mu, LAMBDA_SESGOS)
personal_nuevo = predecir(u_nuevo, s.V)[0]
r_hat_nuevo = predecir_con_sesgos(u_nuevo, s.V, b_nuevo, s.c, s.mu)[0]

candidatos = [j for j in np.argsort(-personal_nuevo, kind="stable") if not m_nuevo[0, j]][:TOP_N]
print(f"b = {b_nuevo[0]:+.3f} (calificás {'más alto' if b_nuevo[0] > 0 else 'más bajo'} que el promedio)\n")
print(f"Top-{TOP_N} para vos:")
for posicion, j in enumerate(candidatos, 1):
    print(f"{posicion:>3}. {titulos_por_indice[j]}  ({r_hat_nuevo[j]:.2f})")

## 15. Evaluación con entrenamiento y prueba

Hasta acá los modelos se entrenaron con todas las calificaciones de $\Omega$, así que no hay forma de
saber qué tan bien predicen una calificación que no vieron. Para medirlo, $\Omega$ se divide en dos
conjuntos disjuntos:

- **entrenamiento** $\Omega_E$, con el que se ajustan $U$ y $V$;
- **prueba** $\Omega_P$, que el entrenamiento nunca ve y se usa solo para medir el error.

La división se hace **por usuario**: de cada fila se reserva al azar una fracción `PROPORCION_PRUEBA`
de sus calificaciones para prueba, pero nunca tantas que le queden menos de $k$ en entrenamiento
(si no, su sistema de ALS sería singular). El error se mide con la raíz del error cuadrático medio,

$$\mathrm{RMSE}_{\Omega'} = \sqrt{\frac{1}{\lvert\Omega'\rvert} \sum_{(i,j) \in \Omega'} \left(r_{ij} - \hat r_{ij}\right)^2},$$

que está en las mismas unidades que las calificaciones (estrellas).

Antes de dividir, la tabla muestra cómo achica el filtro por `UMBRAL` a la matriz en cada pasada,
hasta que deja de cambiar.

In [ ]:
def pasadas_del_filtro(M: np.ndarray, umbral: int) -> list[tuple[int, int, int, int, float]]:
    "(pasada, usuarios, películas, calificaciones, densidad) después de cada pasada del filtro; la 0 es sin filtrar."
    filas = np.arange(M.shape[0])
    columnas = np.arange(M.shape[1])
    pasadas = []
    while True:
        submascara = M[np.ix_(filas, columnas)]
        pasadas.append((len(pasadas), len(filas), len(columnas), int(submascara.sum()),
                        submascara.sum() / submascara.size))
        filas_nuevas = filas[submascara.sum(axis=1) >= umbral]
        columnas_nuevas = columnas[submascara.sum(axis=0) >= umbral]
        if len(filas_nuevas) == len(filas) and len(columnas_nuevas) == len(columnas):
            return pasadas
        filas, columnas = filas_nuevas, columnas_nuevas


def dividir_por_usuario(M: np.ndarray, proporcion_prueba: float, k: int,
                        generador: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    """Divide Ω en entrenamiento y prueba, fila por fila.

    De cada usuario elige al azar round(proporcion_prueba · |Ωᵢ|) de sus calificaciones para prueba,
    pero nunca tantas que le queden menos de k en entrenamiento. Devuelve (M_ent, M_pru), disjuntas
    y con unión M.
    """
    M_ent = M.copy()
    M_pru = np.zeros_like(M)
    for i in range(M.shape[0]):
        observadas = np.where(M[i])[0]
        n_prueba = min(int(round(proporcion_prueba * len(observadas))), max(len(observadas) - k, 0))
        elegidas = generador.choice(observadas, size=n_prueba, replace=False)
        M_ent[i, elegidas] = False
        M_pru[i, elegidas] = True
    return M_ent, M_pru


def rmse(R: np.ndarray, R_hat: np.ndarray, mascara: np.ndarray) -> float:
    "RMSE = √(SCE / |mascara|), solo sobre las posiciones True de `mascara`."
    error = np.where(mascara, R - R_hat, 0.0)
    return float(np.sqrt((error**2).sum() / mascara.sum()))


crudo = cargar_calificaciones(RUTA_MOVIELENS / "ratings.csv")
print(f"Pasadas del filtro con umbral={UMBRAL}\n")
print(f"{'pasada':>6}{'usuarios':>10}{'películas':>11}{'calificaciones':>16}{'densidad':>10}")
for pasada, usuarios, peliculas, calificaciones, densidad in pasadas_del_filtro(crudo.M, UMBRAL):
    print(f"{pasada:>6}{usuarios:>10}{peliculas:>11}{calificaciones:>16}{densidad:>10.2%}")

M_ent, M_pru = dividir_por_usuario(M, PROPORCION_PRUEBA, K, np.random.default_rng(SEMILLA))
R_ent = np.where(M_ent, R, np.nan)
print(f"\nEntrenamiento: {int(M_ent.sum())} calificaciones. Prueba: {int(M_pru.sum())}.")
print(f"Mínimo en entrenamiento: {int(M_ent.sum(axis=1).min())} por usuario, "
      f"{int(M_ent.sum(axis=0).min())} por película. "
      f"Películas sin ninguna calificación en prueba: {int((M_pru.sum(axis=0) == 0).sum())}.")

### ALS y descenso de gradiente sobre entrenamiento

Los dos métodos se entrenan solo con $\Omega_E$, otra vez desde el mismo $U_0, V_0$. Como referencia
hay dos **líneas de base** que no factorizan nada: predecir siempre el promedio de entrenamiento y
predecir para cada usuario su propio promedio. Un modelo que no mejora el RMSE de prueba de estas
líneas de base no está aprendiendo nada útil.

La columna *recortado* es el RMSE de prueba después de llevar cada $\hat r_{ij}$ al rango posible de
las calificaciones, $[0{,}5;\ 5]$. La última columna es el mayor $\lvert\hat r_{ij}\rvert$ fuera de $\Omega$.

In [ ]:
def evaluar(R_hat: np.ndarray) -> tuple[float, float, float, float]:
    "RMSE en entrenamiento, en prueba, en prueba recortando r̂ a [0,5; 5] y máximo |r̂| fuera de Ω."
    return (rmse(R, R_hat, M_ent), rmse(R, R_hat, M_pru),
            rmse(R, np.clip(R_hat, 0.5, 5.0), M_pru), float(np.abs(R_hat[~M]).max()))


def entrenar_sobre_entrenamiento(k: int, metodo: str) -> ResultadoEntrenamiento:
    "ALS o descenso de gradiente sobre Ω_E, desde U0 y V0 generados con SEMILLA."
    U0_ent, V0_ent = inicializar_factores(m, n, k, ESCALA_INICIALIZACION, np.random.default_rng(SEMILLA))
    if metodo == "ALS":
        return entrenar_als(R_ent, M_ent, U0_ent, V0_ent, epsilon=EPSILON, max_iter=MAX_ITER)
    return entrenar_gd(R_ent, M_ent, U0_ent, V0_ent, eta=ETA, epsilon=EPSILON, max_iter=MAX_ITER)


def imprimir_fila(nombre: str, iteraciones: str, R_hat: np.ndarray) -> None:
    rmse_ent, rmse_pru, rmse_recortado, maximo_fuera = evaluar(R_hat)
    print(f"{nombre:<24}{iteraciones:>7}{rmse_ent:>11.4f}{rmse_pru:>13.4f}{rmse_recortado:>11.4f}{maximo_fuera:>16.2f}")


modelos_ent = {metodo: entrenar_sobre_entrenamiento(K, metodo) for metodo in ("ALS", "GD")}

mu_ent = float(np.nanmean(R_ent))
print(f"\n{'modelo':<24}{'iter.':>7}{'RMSE ent.':>11}{'RMSE prueba':>13}{'recortado':>11}{'máx |r̂| fuera':>16}")
imprimir_fila("promedio global", "—", np.full((m, n), mu_ent))
imprimir_fila("promedio del usuario", "—", np.repeat(np.nanmean(R_ent, axis=1)[:, None], n, axis=1))
for metodo, nombre in (("ALS", f"ALS (k={K})"), ("GD", f"Desc. gradiente (k={K})")):
    resultado = modelos_ent[metodo]
    imprimir_fila(nombre, str(resultado.n_iteraciones), predecir(resultado.U, resultado.V))

### Barrido de k

Lo mismo con varios valores de $k$ (`KS_BARRIDO`). Sin regularización, más dimensiones bajan el error
de entrenamiento pero no necesariamente el de prueba, y las estimaciones fuera de $\Omega$ pueden
dispararse. Si algún entrenamiento falla (sistema singular o divergencia), la tabla muestra el error en
vez de cortar el notebook. Durante el barrido se ocultan los mensajes de progreso.

In [ ]:
nivel_anterior = LOGGER.level
LOGGER.setLevel(logging.ERROR)  # sin el progreso ni los avisos de cada entrenamiento
try:
    print(f"{'k':>3}  {'método':<8}{'iter.':>7}{'RMSE ent.':>11}{'RMSE prueba':>13}{'máx |r̂| fuera':>16}")
    for k in KS_BARRIDO:
        for metodo in ("ALS", "GD"):
            try:
                resultado = entrenar_sobre_entrenamiento(k, metodo)
            except (DivergenciaError, SistemaSingularError) as error:
                print(f"{k:>3}  {metodo:<8}  {error}")
                continue
            rmse_ent, rmse_pru, _, maximo_fuera = evaluar(predecir(resultado.U, resultado.V))
            print(f"{k:>3}  {metodo:<8}{resultado.n_iteraciones:>7}{rmse_ent:>11.4f}{rmse_pru:>13.4f}{maximo_fuera:>16.2f}")
finally:
    LOGGER.setLevel(nivel_anterior)

### Figura 1: convergencia sobre entrenamiento

$f$ por iteración para los dos métodos entrenados sobre $\Omega_E$, en escala logarítmica. El recuadro
amplía las primeras iteraciones, donde se ve la diferencia: ALS baja casi todo en los primeros pasos y el
descenso de gradiente avanza de a poco.

In [ ]:
import re

from matplotlib.ticker import FuncFormatter

AZUL, NARANJA = "#2a78d6", "#eb6834"
TEXTO, TEXTO_SECUNDARIO, GRILLA = "#0b0b0b", "#52514e", "#e4e3df"
ESTILO_FIGURAS = {"font.size": 10, "axes.edgecolor": TEXTO_SECUNDARIO, "axes.labelcolor": TEXTO,
                  "xtick.color": TEXTO_SECUNDARIO, "ytick.color": TEXTO_SECUNDARIO}
ITERACIONES_RECUADRO = 30


def _miles(x: float, _posicion: object = None) -> str:
    "Entero con punto de miles: 100000 -> '100.000'."
    return f"{x:,.0f}".replace(",", ".")


def _coma(x: float, _posicion: object = None) -> str:
    "Un decimal con coma: 0.5 -> '0,5'."
    return f"{x:.1f}".replace(".", ",")


def _sin_etiqueta(_x: float, _posicion: object = None) -> str:
    return ""


def _estilo(ax: plt.Axes) -> None:
    "Ejes recesivos: sin bordes superior y derecho, grilla clara detrás."
    for borde in ("top", "right"):
        ax.spines[borde].set_visible(False)
    ax.grid(True, color=GRILLA, linewidth=0.8)
    ax.set_axisbelow(True)


def graficar_convergencia(historiales: dict[str, list[float]]) -> None:
    "f por iteración para ALS y GD, escala log, con recuadro de las primeras iteraciones."
    fig, ax = plt.subplots(figsize=(8, 4.8))
    for metodo, color in (("ALS", AZUL), ("GD", NARANJA)):
        historial = historiales[metodo]
        ultimo = len(historial) - 1
        ax.plot(range(len(historial)), historial, color=color, linewidth=2,
                marker="o" if metodo == "ALS" else None, markersize=4)
        nombre = "ALS" if metodo == "ALS" else "Descenso de gradiente"
        ax.annotate(f"{nombre}: {ultimo} iteraciones", xy=(ultimo, historial[-1]),
                    xytext=(ultimo + (8 if metodo == "ALS" else -10),
                            historial[-1] * (1.9 if metodo == "ALS" else 1.5)),
                    ha="left" if metodo == "ALS" else "right", color=TEXTO, fontsize=9,
                    arrowprops=dict(arrowstyle="-", color=TEXTO_SECUNDARIO, lw=0.8))
    ax.set_yscale("log")
    ax.yaxis.set_major_formatter(FuncFormatter(_miles))
    ax.yaxis.set_minor_formatter(FuncFormatter(_sin_etiqueta))
    ax.set_yticks([20000, 50000, 100000, 200000, 350000])

    recuadro = ax.inset_axes([0.42, 0.42, 0.55, 0.5])
    for metodo, color in (("ALS", AZUL), ("GD", NARANJA)):
        historial = historiales[metodo][: ITERACIONES_RECUADRO + 1]
        recuadro.plot(range(len(historial)), historial, color=color, linewidth=2, marker="o", markersize=3)
    recuadro.set_yscale("log")
    recuadro.set_ylim(15000, 400000)
    recuadro.set_yticks([20000, 50000, 100000, 300000])
    recuadro.yaxis.set_major_formatter(FuncFormatter(_miles))
    recuadro.yaxis.set_minor_formatter(FuncFormatter(_sin_etiqueta))
    recuadro.set_title(f"primeras {ITERACIONES_RECUADRO} iteraciones", fontsize=9, color=TEXTO_SECUNDARIO,
                       backgroundcolor="white")
    recuadro.tick_params(labelsize=8)
    _estilo(recuadro)

    ax.set_xlabel("iteración t")
    ax.set_ylabel("f(U, V) sobre Ω de entrenamiento (escala log)")
    _estilo(ax)
    fig.tight_layout()
    plt.show()


with plt.rc_context(ESTILO_FIGURAS):
    graficar_convergencia({metodo: resultado.historial_f for metodo, resultado in modelos_ent.items()})

### Figura 2: las películas en el plano de V

Cada punto es la fila $v_j$ de una de las `N_POPULARES` películas más calificadas, en el $V$ de ALS
entrenado sobre $\Omega_E$. Las coordenadas no tienen un significado asignado de antemano: películas
cercanas son películas que los mismos usuarios califican parecido. Con $k > 2$ se dibujan solo las
dos primeras coordenadas.

Cada etiqueta prueba posiciones alrededor de su punto, de la más cercana a la más lejana, y se queda con
la primera que no pisa otra etiqueta, ningún punto ni el borde. Las que quedan lejos llevan una línea guía.

In [ ]:
def _titulo_corto(titulo: str) -> str:
    """Acorta un título de MovieLens para la etiqueta del gráfico.

    Saca el año y los "a.k.a.", pasa "Godfather, The" a "The Godfather" y abrevia Star Wars y
    El señor de los anillos.
    """
    titulo = re.sub(r"\s*\(a\.k\.a\.[^)]*\)", "", titulo)
    titulo = re.sub(r"\s*\((Indiana Jones[^)]*)\)", "", titulo)
    articulo = re.match(r"(.*?), (The|A)( \(\d{4}\))$", titulo)
    if articulo:
        titulo = f"{articulo.group(2)} {articulo.group(1)}{articulo.group(3)}"
    titulo = re.sub(r" \(\d{4}\)$", "", titulo)
    titulo = titulo.replace("Star Wars: Episode ", "Star Wars ").replace("Lord of the Rings: The ", "LOTR: ")
    for subtitulo in (" - A New Hope", " - The Empire Strikes Back", " - Return of the Jedi"):
        titulo = titulo.replace(subtitulo, "")
    return titulo


def graficar_dispersion(x: np.ndarray, y: np.ndarray, etiquetas: list[str]) -> None:
    "Puntos (x, y) con su etiqueta, ubicadas para que no se pisen entre sí ni con los puntos."
    fig, ax = plt.subplots(figsize=(9, 9))
    ax.scatter(x, y, s=28, color=AZUL, edgecolor="white", linewidth=1, zorder=3)
    ax.set_xlabel("primera coordenada de vⱼ")
    ax.set_ylabel("segunda coordenada de vⱼ")
    ax.xaxis.set_major_formatter(FuncFormatter(_coma))
    ax.yaxis.set_major_formatter(FuncFormatter(_coma))
    ax.set_xlim(x.min() - 0.12, x.max() + 0.3)
    ax.set_ylim(y.min() - 0.08, y.max() + 0.08)
    _estilo(ax)
    fig.tight_layout()

    renderer = fig.canvas.get_renderer()
    marco = ax.get_window_extent(renderer)
    puntos_pantalla = ax.transData.transform(np.column_stack([x, y]))
    radio = 4 * fig.dpi / 72  # medio diámetro del marcador, con margen
    # Desplazamientos candidatos en puntos tipográficos: (dx, dy, alineación horizontal)
    candidatos = [(6, 0, "left"), (-6, 0, "right")]
    for distancia in (9, 14, 20, 27, 35, 44):
        candidatos += [(6, distancia, "left"), (6, -distancia, "left"),
                       (-6, distancia, "right"), (-6, -distancia, "right")]
    puestas: list = []

    def choca(caja) -> bool:
        if caja.x0 < marco.x0 or caja.x1 > marco.x1 or caja.y0 < marco.y0 or caja.y1 > marco.y1:
            return True
        if any(caja.overlaps(otra) for otra in puestas):
            return True
        return any(caja.x0 - radio < px < caja.x1 + radio and caja.y0 - radio < py < caja.y1 + radio
                   for px, py in puntos_pantalla)

    # Primero las zonas más densas, que tienen menos lugares libres.
    vecinos = [np.sum(np.hypot(*(puntos_pantalla - p).T) < 60) for p in puntos_pantalla]
    for i in sorted(range(len(x)), key=lambda i: -vecinos[i]):
        for dx, dy, alineacion in candidatos:
            texto = ax.annotate(etiquetas[i], xy=(x[i], y[i]), xytext=(dx, dy), textcoords="offset points",
                                fontsize=7, color=TEXTO, va="center", ha=alineacion)
            caja = texto.get_window_extent(renderer).expanded(1.04, 1.15)
            if not choca(caja):
                break
            texto.remove()
        else:
            LOGGER.warning("sin lugar libre para la etiqueta %r; queda en la última posición", etiquetas[i])
            texto = ax.annotate(etiquetas[i], xy=(x[i], y[i]), xytext=(dx, dy), textcoords="offset points",
                                fontsize=7, color=TEXTO, va="center", ha=alineacion)
            caja = texto.get_window_extent(renderer)
        if dy != 0:
            ax.annotate("", xy=(x[i], y[i]), xytext=(dx, dy), textcoords="offset points",
                        arrowprops=dict(arrowstyle="-", color="#a8a7a2", lw=0.5, shrinkA=0, shrinkB=3))
        puestas.append(caja)
    plt.show()


V_ent = modelos_ent["ALS"].V
if V_ent.shape[1] < 2:
    print(f"Con K={K} no hay plano para dibujar: hace falta K ≥ 2.")
else:
    populares = np.argsort(-M.sum(axis=0), kind="stable")[:N_POPULARES]
    with plt.rc_context(ESTILO_FIGURAS):
        graficar_dispersion(V_ent[populares, 0], V_ent[populares, 1],
                            [_titulo_corto(titulos_por_indice[j]) for j in populares])

### ¿Aprenden lo mismo ALS y descenso de gradiente?

Los dos métodos minimizan la misma $f$ desde el mismo punto, pero no tienen por qué llegar a los mismos
$U$ y $V$: la factorización no es única (cualquier $U Q$, $V Q^{-\top}$ con $Q$ invertible da el mismo
$\hat R$). Por eso se comparan las **predicciones**, además de las columnas de $V$.

Al final se mira si la dirección diagonal del plano, $v_{j1} + v_{j2}$, sigue al promedio de cada
película: si es así, una buena parte de lo que aprendió $V$ es la calidad general de la película y no
gustos personales, que es justo el problema que resuelve el modelo con sesgos de la sección 13.

In [ ]:
def correlacion(x: np.ndarray, y: np.ndarray) -> float:
    "Coeficiente de correlación de Pearson entre dos vectores."
    return float(np.corrcoef(x, y)[0, 1])


als_ent, gd_ent = modelos_ent["ALS"], modelos_ent["GD"]
for columna in range(K):
    print(f"correlación de la columna {columna + 1} de V, ALS vs. GD: "
          f"{correlacion(als_ent.V[:, columna], gd_ent.V[:, columna]):.4f}")

R_hat_als_ent = predecir(als_ent.U, als_ent.V)
R_hat_gd_ent = predecir(gd_ent.U, gd_ent.V)
print(f"\ncorrelación de las predicciones ALS vs. GD sobre Ω_P:   {correlacion(R_hat_als_ent[M_pru], R_hat_gd_ent[M_pru]):.4f}")
print(f"RMSE entre las predicciones de ALS y de GD sobre Ω_P:   {rmse(R_hat_als_ent, R_hat_gd_ent, M_pru):.4f}")
print(f"correlación de las predicciones en todas las entradas: {correlacion(R_hat_als_ent.ravel(), R_hat_gd_ent.ravel()):.4f}")

if K >= 2:
    promedio_pelicula = np.nanmean(R, axis=0)
    print(f"\ncorrelación entre las dos primeras columnas de V (ALS): {correlacion(als_ent.V[:, 0], als_ent.V[:, 1]):.4f}")
    print(f"correlación de v_j1 + v_j2 con el promedio de cada película: "
          f"{correlacion(als_ent.V[:, 0] + als_ent.V[:, 1], promedio_pelicula):.4f}")